# Verbal Uncertainty Feature Calibration — Colab Pipeline

**Paper:** *Calibrating Verbal Uncertainty as a Linear Feature to Reduce Hallucinations* (Ji et al., 2025)  
**Original repository:** https://github.com/facebookresearch/verbal_uncertainty_feature_calibration

### Differences from original
| Original | This notebook |
|---|---|
| SLURM + `submitit` | Direct `python script.py` calls |
| `squeue`-based vLLM discovery | Local vLLM subprocess |
| Llama-3.1-**70B** judge | Mistral-7B (served under alias Llama-70B) |
| TriviaQA (factual) | **NQ-Open** (Wikipedia-based, less factual) |
| 3 datasets × 3 splits × 3 models | 1 dataset × **train/val/test** × 1 model |

### Run modes
| Parameter | Value | Description |
|---|---|---|
| `DRY_RUN = False` | full run | train=10k, val=1k, test=1k (~4–6 hours) |
| `DRY_RUN = True` | test run | `DRY_RUN_SIZE` rows/split (~10–20 min) |

> **Note:** With `DRY_RUN=True` the results (AUROC, VU) will not be representative —
> it is only a pipeline sanity check.

**Recommended environment:** Colab Pro / **A100 40 GB**

## 0. Check GPU

In [1]:
import subprocess, torch

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {vram:.1f} GB")
    if vram < 20:
        print("WARNING: T4 (15 GB) — генерация и vLLM не могут работать одновременно.")
    else:
        print("OK: A100 40 GB.")
else:
    raise RuntimeError("GPU не найден. Runtime → Change runtime type → GPU.")

Mon Mar 23 14:13:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P0             51W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. Clone and install dependencies

In [2]:
import os

REPO_DIR = "/content/verbal_uncertainty_feature_calibration"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/facebookresearch/verbal_uncertainty_feature_calibration {REPO_DIR}
else:
    print("Репозиторий уже клонирован.")

%cd {REPO_DIR}

Cloning into '/content/verbal_uncertainty_feature_calibration'...
remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 87 (delta 19), reused 44 (delta 15), pack-reused 27 (from 1)
Receiving objects: 100% (87/87), 2.76 MiB | 35.77 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/verbal_uncertainty_feature_calibration


In [3]:
!pip install -q \
    transformers==4.48.0 \
    accelerate \
    datasets \
    evaluate==0.4.3 \
    peft==0.13.2 \
    safetensors \
    tokenizers \
    einops \
    jsonlines \
    tenacity \
    openai \
    scikit-learn \
    scipy \
    pandas \
    tqdm \
    submitit

!pip install -q vllm

print("Зависимости установлены.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 107.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.2/433.2 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8

## 2. All patches (run once)

All fixes are applied on clone or when re-running the notebook:

| Patch | File | Purpose |
|---|---|---|
| 1 | `src/eval_utils.py` | Remove SLURM `squeue`, return local vLLM URL |
| 2 | `sem_uncertainty/generate_answers.py` | Move tensors to GPU (`BatchEncoding.to(device)`) |
| 3 | `datasets/merge.py` | Revert old restriction `['test']` → all `['test','val','train']` |
| 4 | `calibration/merge_vuf.py` | Same for merge_vuf |
| 5 | `detection/LogisticRegression.py` | Revert `train_split='test'` → correct `train_split='train'` |
| 6 | `calibration/causal.py` | Fix `hf_device_map` + remove `assert False` + add `makedirs` |

In [24]:
import os

REPO_DIR = "/content/verbal_uncertainty_feature_calibration"

# ── 1. src/eval_utils.py ─────────────────────────────────────────────────────
with open(f"{REPO_DIR}/src/eval_utils.py", "w") as f:
    f.write('''
import os, sys, openai
from tqdm.contrib.concurrent import thread_map

current_dir = os.path.dirname(os.path.abspath(__file__))
root_path = os.path.dirname(current_dir)
sys.path.append(f"{root_path}/sem_uncertainty/")
from semantic_entropy.huggingface_models import HuggingfaceModel


class VLLM:
    def __init__(self, name, max_new_tokens):
        self.port = name
        self.max_tokens = max_new_tokens
        # OpenRouter uses a different model id casing than local vLLM.
        self.openrouter_model_id = os.environ.get(
            "OPENROUTER_MODEL_ID",
            "meta-llama/llama-3.1-70b-instruct",
        )
        self.vllm_model_id = "meta-llama/Llama-3.1-70B-Instruct"

    def predict(self, prompt, temperature, output_hidden_states=False):
        is_openrouter = ("openrouter.ai" in self.port) or ("openrouter" in self.port)
        api_key = os.environ.get("OPENROUTER_API_KEY") if is_openrouter else "NOT A REAL KEY"
        model_id = self.openrouter_model_id if is_openrouter else self.vllm_model_id
        client = openai.OpenAI(base_url=self.port, api_key=api_key)
        resp = client.chat.completions.create(
            model=model_id,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=self.max_tokens,
            temperature=temperature,
        )
        return resp.choices[0].message.content, "", ""

    def batch_predict(self, batch_prompts, temperature, output_hidden_states=False):
        return thread_map(
            lambda p: self.predict(p, temperature=temperature),
            batch_prompts, max_workers=200, desc="vllm",
        )


def get_available_servers():
    vllm_url = os.environ.get("VLLM_URL", "http://localhost:8000/v1")
    return {
        "meta-llama/Llama-3.1-70B-Instruct": {
            "server_urls": [vllm_url],
            "job_ids": ["0"],
        }
    }
''')
print("[1/6] eval_utils.py OK")

# ── 2. generate_answers.py — BatchEncoding.to(device) ────────────────────────
path = f"{REPO_DIR}/sem_uncertainty/generate_answers.py"
code = open(path).read()
old = "        inputs = prepare_inputs(tokenizer, batch_local_prompt)\n\n        with torch.no_grad():"
new = "        inputs = prepare_inputs(tokenizer, batch_local_prompt)\n        inputs = inputs.to(device)\n\n        with torch.no_grad():"
if old in code:
    open(path, "w").write(code.replace(old, new))
    print("[2/6] generate_answers.py OK")
else:
    print("[2/6] generate_answers.py — уже пропатчен")

# ── 3. datasets/merge.py — восстановить все сплиты ───────────────────────────
# (Откат старого патча, который ограничивал только ['test'])
path = f"{REPO_DIR}/datasets/merge.py"
code = open(path).read()
if "for split in ['test']:" in code:
    open(path, "w").write(code.replace(
        "for split in ['test']:",
        "for split in ['test', 'val', 'train']:"
    ))
    print("[3/6] datasets/merge.py — все сплиты восстановлены")
else:
    print("[3/6] datasets/merge.py — OK (уже все сплиты)")

# ── 4. calibration/merge_vuf.py — восстановить все сплиты ────────────────────
path = f"{REPO_DIR}/calibration/merge_vuf.py"
code = open(path).read()
if "for split in ['test']:" in code:
    open(path, "w").write(code.replace(
        "for split in ['test']:",
        "for split in ['val', 'test', 'train']:"
    ))
    print("[4/6] merge_vuf.py — все сплиты восстановлены")
else:
    print("[4/6] merge_vuf.py — OK (уже все сплиты)")

# ── 5. detection/LogisticRegression.py — обучение на train ───────────────────
# (Откат старого патча train_split = 'test' → правильное train_split = 'train')
path = f"{REPO_DIR}/detection/LogisticRegression.py"
code = open(path).read()
if "train_split = 'test'" in code:
    open(path, "w").write(code.replace("train_split = 'test'", "train_split = 'train'"))
    print("[5/6] LogisticRegression.py — train_split='train' восстановлен")
else:
    print("[5/6] LogisticRegression.py — OK")

# ── 6. calibration/causal.py — три фикса ─────────────────────────────────────
path = f"{REPO_DIR}/calibration/causal.py"
code = open(path).read()

# 6a: hf_device_map для одного GPU
old_a = '        device_idx_l = model.hf_device_map[f"model.layers.{l}"]'
new_a = '        device_idx_l = model.hf_device_map.get(f"model.layers.{l}", model.hf_device_map.get("", 0))'
code = code.replace(old_a, new_a)

# 6b: assert False в forward hook (новые версии transformers не возвращают tuple)
old_b = "                else:\n                    assert False\n                    if outputs.shape[1] > 1:\n                        outputs += h_feature_l * alpha\n\n                    return outputs"
new_b = "                else:\n                    if outputs.shape[1] > 1:\n                        outputs = outputs + h_feature_l * alpha\n                    return outputs"
code = code.replace(old_b, new_b)

# 6c: makedirs для output_dir
old_c = '    else:\n        output_dir = f"outputs/{dataset}/{model_name}/{prompt_type}/{split}"\n    if args.run_certain:'
new_c = '    else:\n        output_dir = f"outputs/{dataset}/{model_name}/{prompt_type}/{split}"\n    os.makedirs(output_dir, exist_ok=True)\n    if args.run_certain:'
code = code.replace(old_c, new_c)

open(path, "w").write(code)
print("[6/6] causal.py OK")

# ── 7. verbal_uncertainty/vu_llm_judge.py — OpenRouter support ─────────────
path = f"{REPO_DIR}/verbal_uncertainty/vu_llm_judge.py"
code = open(path).read()
if "api_key=\"NOT A REAL KEY\"" in code:
    code = code.replace(
        "api_key=\"NOT A REAL KEY\"",
        "api_key=os.environ.get(\"OPENROUTER_API_KEY\") if \"openrouter.ai\" in judge_model else \"NOT A REAL KEY\"",
    )
    code = code.replace(
        "model='meta-llama/Llama-3.1-70B-Instruct'",
        "model=os.environ.get(\"OPENROUTER_MODEL_ID\",\"meta-llama/llama-3.1-70b-instruct\") if \"openrouter.ai\" in judge_model else 'meta-llama/Llama-3.1-70B-Instruct'",
    )
    open(path, "w").write(code)
    print("[7/9] vu_llm_judge.py OK")
else:
    print("[7/9] vu_llm_judge.py — уже пропатчен")

# ── 8. src/refusal.py — OpenRouter support ────────────────────────────────
path = f"{REPO_DIR}/src/refusal.py"
code = open(path).read()
if "api_key=\"NOT A REAL KEY\"" in code:
    code = code.replace(
        "api_key=\"NOT A REAL KEY\"",
        "api_key=os.environ.get(\"OPENROUTER_API_KEY\") if \"openrouter.ai\" in str(port) else \"NOT A REAL KEY\"",
    )
    code = code.replace(
        "model='meta-llama/Llama-3.1-70B-Instruct'",
        "model=os.environ.get(\"OPENROUTER_MODEL_ID\",\"meta-llama/llama-3.1-70b-instruct\") if \"openrouter.ai\" in str(port) else 'meta-llama/Llama-3.1-70B-Instruct'",
    )
    open(path, "w").write(code)
    print("[8/9] refusal.py OK")
else:
    print("[8/9] refusal.py — уже пропатчен")

# ── 9. sem_uncertainty/semantic_entropy/semantic_entropy.py — OpenRouter ─
path = f"{REPO_DIR}/sem_uncertainty/semantic_entropy/semantic_entropy.py"
code = open(path).read()
if "api_key=\"NOT A REAL KEY\"" in code:
    code = code.replace(
        "api_key=\"NOT A REAL KEY\"",
        "api_key=os.environ.get(\"OPENROUTER_API_KEY\") if \"openrouter.ai\" in self.port else \"NOT A REAL KEY\"",
    )
    code = code.replace(
        "model=self.name",
        "model=os.environ.get(\"OPENROUTER_MODEL_ID\",\"meta-llama/llama-3.1-70b-instruct\") if \"openrouter.ai\" in self.port else self.name",
    )
    open(path, "w").write(code)
    print("[9/9] semantic_entropy.py OK")
else:
    print("[9/9] semantic_entropy.py — уже пропатчен")

print("\nВсе патчи применены.")

[1/6] eval_utils.py OK
[2/6] generate_answers.py — уже пропатчен
[3/6] datasets/merge.py — OK (уже все сплиты)
[4/6] merge_vuf.py — OK (уже все сплиты)
[5/6] LogisticRegression.py — OK
[6/6] causal.py OK
[7/9] vu_llm_judge.py — уже пропатчен
[8/9] refusal.py — уже пропатчен
[9/9] semantic_entropy.py — уже пропатчен

Все патчи применены.


## 3. Configuration

In [34]:
import os, sys, subprocess

REPO_DIR    = "/content/verbal_uncertainty_feature_calibration"

# ── Режим запуска ──────────────────────────────────────────────────────────────
# DRY_RUN=True  → быстрый тест: обрезает каждый split до DRY_RUN_SIZE строк
# DRY_RUN=False → полный прогон (train=10000, val=1000, test=1000)
DRY_RUN      = True
DRY_RUN_SIZE = 500     # строк на split при DRY_RUN=True

# QA-модель (генерирует ответы напрямую через HuggingFace)
QA_MODEL    = "Mistral-7B-Instruct-v0.3"
QA_HF_ID    = "mistralai/Mistral-7B-Instruct-v0.3"

# Judge-модель (подаётся через vLLM под алиасом Llama-70B —
# все скрипты хардкодят это имя при обращении к API)
JUDGE_HF_ID = "mistralai/Mistral-7B-Instruct-v0.3"
JUDGE_ALIAS = "qwen/qwen3-235b-a22b-2507"

# Датасет: NQ-Open — Natural Questions Open
# train=10000, val=1000, test=1000 (из google-research-datasets/nq_open)
DATASET     = "nq_open"
SPLITS      = ["train", "test"]   # честные три сплита

# Judge/entailment model endpoint (OpenAI-compatible base_url)
USE_OPENROUTER = True
# Load from environment (Colab Secrets, or export before running the notebook)
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_MODEL_ID = "meta-llama/llama-3.1-70b-instruct"

VLLM_PORT   = 8000
VLLM_LOG    = "/tmp/vllm.log"

if USE_OPENROUTER:
    if not OPENROUTER_API_KEY:
        raise RuntimeError("Set OPENROUTER_API_KEY in runtime env (OpenRouter key).")
    os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
    os.environ["OPENROUTER_MODEL_ID"] = OPENROUTER_MODEL_ID
    VLLM_URL = OPENROUTER_BASE_URL
else:
    VLLM_URL = f"http://localhost:{VLLM_PORT}/v1"

os.environ["VLLM_URL"] = VLLM_URL
os.environ["HF_HOME"]  = "/content/.cache/huggingface"
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# ── Хелпер: запуск subprocess с потоковым выводом логов ───────────────────────
def run(cmd, cwd=None):
    """Запускает процесс, выводя stdout+stderr построчно в реальном времени.
    Бросает subprocess.CalledProcessError при ненулевом коде возврата."""
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        cwd=cwd,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)

print(f"QA модель  : {QA_HF_ID}")
print(f"Judge      : {JUDGE_HF_ID} → алиас {JUDGE_ALIAS}")
print(f"Датасет    : {DATASET},  splits={SPLITS}")
if DRY_RUN:
    print(f"DRY_RUN    : ON — {DRY_RUN_SIZE} строк/split (только для проверки пайплайна)")
else:
    print(f"DRY_RUN    : OFF — полный датасет")
print(f"vLLM URL   : {VLLM_URL}")

QA модель  : mistralai/Mistral-7B-Instruct-v0.3
Judge      : mistralai/Mistral-7B-Instruct-v0.3 → алиас qwen/qwen3-235b-a22b-2507
Датасет    : nq_open,  splits=['train', 'test']
DRY_RUN    : ON — 500 строк/split (только для проверки пайплайна)
vLLM URL   : https://openrouter.ai/api/v1


## 4. HuggingFace Login

In [11]:
from huggingface_hub import login
login()  # вставьте токен с read-доступом

## 5. Download dataset

**Paper, Appendix B** — NQ-Open: 10,000 train + 1,000 test + 1,000 val from `google-research-datasets/nq_open`.  
Wikipedia-based QA. Answers are short text spans.  
Less factual than TriviaQA: the model expresses uncertainty more often (Table 6 in the paper).

In [17]:
!{sys.executable} datasets/download_dataset.py --dataset_name {DATASET}

In [18]:
# ── Dry run: обрезаем CSV датасета до DRY_RUN_SIZE строк ─────────────────────
# При DRY_RUN=False — просто выводим размеры.
# При DRY_RUN=True  — усекаем train/val/test.csv, чтобы ускорить все шаги.
# Для возврата к полному датасету перезапустите ячейку скачивания (раздел 5).
import pandas as pd

sampled_dir = f"{REPO_DIR}/datasets/{DATASET}/sampled"
print("Размер датасета после загрузки:")
for split in SPLITS:
    path = f"{sampled_dir}/{split}.csv"
    if os.path.exists(path):
        n = len(pd.read_csv(path))
        print(f"  {split:6s}.csv : {n} строк")
    else:
        print(f"  {split:6s}.csv : ОТСУТСТВУЕТ")

if DRY_RUN:
    print(f"\nDRY_RUN=True — обрезаем до {DRY_RUN_SIZE} строк/split:")
    for split in SPLITS:
        path = f"{sampled_dir}/{split}.csv"
        if os.path.exists(path):
            df = pd.read_csv(path)
            n_before = len(df)
            df_small = df.head(DRY_RUN_SIZE)
            df_small.to_csv(path, index=False)
            print(f"  {split:6s}.csv : {n_before} → {len(df_small)} строк")
    print("\nДля возврата к полным данным — перезапустите ячейку скачивания (раздел 5).")
else:
    print("\nDRY_RUN=False — обрабатываем полный датасет.")

Размер датасета после загрузки:
  train .csv : 10000 строк
  test  .csv : 1000 строк

DRY_RUN=True — обрезаем до 500 строк/split:
  train .csv : 10000 → 500 строк
  test  .csv : 1000 → 500 строк

Для возврата к полным данным — перезапустите ячейку скачивания (раздел 5).


---
# Phase 1 — Answer Generation (HuggingFace)

Three steps load the QA model via HuggingFace and run inference directly.  
GPU is fully freed after each cell (subprocess terminates).

### 5.1 Generate 10 answers at T=1.0 (for semantic entropy)

**Details:** Mistral generates 10 answers per question at T=1.0 with nucleus sampling (p=0.9, k=50).  
Output → `sem_uncertainty/outputs/nq_open/sentence/{model}/test_1.0.jsonl`.

**Paper §2.1, Appendix C** — "we input a question and sample 10 sequences using temperature=1 with nucleus sampling"

In [19]:
for split in SPLITS:
    print(f"\n=== generate_answers T=1.0 | split={split} ===")
    run([
        sys.executable, "sem_uncertainty/generate_answers.py",
        "--dataset",     DATASET,
        "--split",       split,
        "--model_name",  QA_MODEL,
        "--prompt_type", "sentence",
        "--temperature", "1.0",
    ], cwd=REPO_DIR)


=== generate_answers T=1.0 | split=train ===
2026-03-23 14:34:11.757865: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-23 14:34:11.776877: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774276451.799468    6126 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774276451.807012    6126 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774276451.825869    6126 computation_placer.cc:177] computation placer already regis

### 5.2 Generate answers with uncertainty prompt (for verbal uncertainty)

**Details:** Mistral generates 10 answers at T=1.0, but with a system prompt encouraging hedging ("If you are uncertain, convey this uncertainty verbally").  
Output → `verbal_uncertainty/outputs/{model}_{dataset}_{split}_uncertainty_1.0.json`.

**Paper §2.2, Appendix A.1** — Answer Generation Prompt for Verbal Uncertainty

In [20]:
for split in SPLITS:
    print(f"\n=== qa_generate (VU prompt) | split={split} ===")
    run([
        sys.executable, "verbal_uncertainty/qa_generate.py",
        "--dataset",                 DATASET,
        "--split",                   split,
        "--model_name",              QA_MODEL,
        "--prompt_method",           "uncertainty",
        "--n_response_per_question", "10",
        "--temperature",             "1.0",
        "--max_new_tokens",          "100",
        "--results_dir",             "outputs",
    ], cwd=REPO_DIR)


=== qa_generate (VU prompt) | split=train ===
2026-03-23 14:40:35.718643: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-23 14:40:35.737709: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774276835.760134    7857 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774276835.767716    7857 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774276835.786768    7857 computation_placer.cc:177] computation placer already regi

### 5.3 Generate most likely answer at T=0.1 (for accuracy evaluation)

**Details:** Mistral generates 1 answer at T=0.1 (greedy decoding) — the model's most likely answer.  
This answer will be compared against ground truth and evaluated for hallucinations.  
Output → `sem_uncertainty/outputs/nq_open/sentence/{model}/test_0.1.jsonl`.

**Paper §2.3, Appendix C** — "generate a single sequence at low temperature (0.1) to estimate the most likely answer"

In [21]:
for split in SPLITS:
    print(f"\n=== generate_answers T=0.1 | split={split} ===")
    run([
        sys.executable, "sem_uncertainty/generate_answers.py",
        "--dataset",     DATASET,
        "--split",       split,
        "--model_name",  QA_MODEL,
        "--prompt_type", "sentence",
        "--temperature", "0.1",
    ], cwd=REPO_DIR)


=== generate_answers T=0.1 | split=train ===
2026-03-23 14:51:16.360940: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-23 14:51:16.379526: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774277476.401492   10666 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774277476.408922   10666 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774277476.427415   10666 computation_placer.cc:177] computation placer already regis

---
# Phase 2 — vLLM Judge

Mistral-7B is launched locally as an OpenAI-compatible server.  
The alias `meta-llama/Llama-3.1-70B-Instruct` is needed because all scripts hardcode this name in API requests.

### 6.0 Start vLLM server

In [36]:
import time, requests, gc, torch

gc.collect()
torch.cuda.empty_cache()

vllm_proc = None

if not USE_OPENROUTER:
    vllm_proc = subprocess.Popen(
        [
            sys.executable, "-m", "vllm.entrypoints.openai.api_server",
            "--model",                  JUDGE_HF_ID,
            "--served-model-name",      JUDGE_ALIAS,
            "--port",                   str(VLLM_PORT),
            "--gpu-memory-utilization", "0.85",
            "--max-model-len",          "4096",
            "--dtype",                  "float16",
        ],
        stdout=open(VLLM_LOG, "w"),
        stderr=subprocess.STDOUT,
        cwd=REPO_DIR,
    )
    print(f"vLLM PID={vllm_proc.pid}. Логи: {VLLM_LOG}")

    for i in range(120):
        try:
            if requests.get(f"http://localhost:{VLLM_PORT}/health", timeout=3).status_code == 200:
                print(f"vLLM готов ({i*5}s)")
                break
        except Exception:
            pass
        print(f"  ожидание {(i+1)*5}s...", end="\r")
        time.sleep(5)
    else:
        print("Сервер не поднялся. Логи:")
        !tail -30 {VLLM_LOG}
else:
    print("USE_OPENROUTER=True — пропускаем запуск vLLM сервера (используем OpenRouter).")

USE_OPENROUTER=True — пропускаем запуск vLLM сервера (используем OpenRouter).


### 6.1 Compute semantic entropy

**Details:** 10 answers per question are clustered by semantic equivalence using NLI (judge via vLLM). Semantic entropy = entropy of the cluster distribution.  
Output → `test_semantic_entropy.pkl`.

**Paper §2.1** — Semantic Entropy formula; Farquhar et al. 2024

In [42]:
import os
os.environ["SEM_ENTROPY_MAX_WORKERS"] = "40"          # чтобы меньше зависало из-за rate limit’ов/очередей
os.environ["OPENROUTER_REQUEST_TIMEOUT"] = "2"      # сек
os.environ["OPENROUTER_MAX_RETRIES"] = "2"            # сколько раз повторять при таймауте/ошибке
os.environ["OPENROUTER_ENTAILMENT_MAX_TOKENS"] = "40" # как раньше

In [43]:
import os
os.environ.pop("OPENROUTER_PROVIDER_ONLY", None)
print("OPENROUTER_PROVIDER_ONLY =", os.environ.get("OPENROUTER_PROVIDER_ONLY"))

OPENROUTER_PROVIDER_ONLY = None


In [44]:
import os, re, importlib, sys

path = "/content/verbal_uncertainty_feature_calibration/sem_uncertainty/semantic_entropy/semantic_entropy.py"
assert os.path.exists(path), f"File not found: {path}"

text = open(path, "r", encoding="utf-8").read()

# 1) Ensure `import time` exists (for time.sleep)
if re.search(r"^\s*import\s+time\s*$", text, flags=re.M) is None:
    # insert after `import logging` if present, else after first import block
    if "import logging" in text:
        text = text.replace("import logging", "import logging\nimport time")
    else:
        text = "import time\n" + text

new_predict = r'''    def predict(self, prompt, temperature):
        is_openrouter = ("openrouter.ai" in self.port) or ("openrouter" in self.port)
        api_key = os.environ.get("OPENROUTER_API_KEY") if is_openrouter else "NOT A REAL KEY"
        model_id = (
            os.environ.get("OPENROUTER_MODEL_ID", "meta-llama/llama-3.1-70b-instruct")
            if is_openrouter
            else self.name
        )

        client = openai.OpenAI(base_url=self.port, api_key=api_key)

        provider_only_raw = os.environ.get("OPENROUTER_PROVIDER_ONLY", "").strip()
        extra_body = None
        if is_openrouter and provider_only_raw:
            # OpenRouter provider routing: OPENROUTER_PROVIDER_ONLY="anthropic,openai"
            providers = [p.strip() for p in provider_only_raw.split(",") if p.strip()]
            extra_body = {"provider": {"only": providers}}

        request_timeout = int(os.environ.get("OPENROUTER_REQUEST_TIMEOUT", "30"))
        max_retries = int(os.environ.get("OPENROUTER_MAX_RETRIES", "3"))

        last_err = None
        for attempt in range(max_retries):
            try:
                try:
                    chat_completion = client.chat.completions.create(
                        model=model_id,
                        messages=[{"role": "user","content": prompt}],
                        max_tokens=int(os.environ.get("OPENROUTER_ENTAILMENT_MAX_TOKENS", "8")),
                        temperature=temperature,
                        extra_body=extra_body,
                        request_timeout=request_timeout,
                    )
                except TypeError:
                    # some SDK versions use `timeout` instead of `request_timeout`
                    chat_completion = client.chat.completions.create(
                        model=model_id,
                        messages=[{"role": "user","content": prompt}],
                        max_tokens=int(os.environ.get("OPENROUTER_ENTAILMENT_MAX_TOKENS", "8")),
                        temperature=temperature,
                        extra_body=extra_body,
                        timeout=request_timeout,
                    )
                return chat_completion.choices[0].message.content
            except Exception as e:
                last_err = e
                if attempt < max_retries - 1:
                    time.sleep(0.5 * (attempt + 1))

        raise last_err
'''

# 2) Replace the whole EntailmentVLLM.predict(...) method by regex until def batch_predict(...)
pattern = r"    def predict\(self, prompt, temperature\):[\s\S]*?\n    def batch_predict"
m = re.search(pattern, text)
assert m is not None, "Could not find predict() method block to replace"

text = re.sub(pattern, new_predict + "\n    def batch_predict", text, count=1)

open(path, "w", encoding="utf-8").write(text)
print("Patched:", path)

# 3) Reload module so the running notebook uses patched code
mod_name = "sem_uncertainty.semantic_entropy.semantic_entropy"
if mod_name in sys.modules:
    importlib.reload(sys.modules[mod_name])
else:
    importlib.import_module(mod_name)

print("Reloaded module:", mod_name)

Patched: /content/verbal_uncertainty_feature_calibration/sem_uncertainty/semantic_entropy/semantic_entropy.py
Reloaded module: sem_uncertainty.semantic_entropy.semantic_entropy


In [48]:
for split in SPLITS:
    print(f"\n=== compute_semantic_entropy | split={split} ===")
    run([
        sys.executable, "sem_uncertainty/compute_semantic_entropy.py",
        "--dataset",    DATASET,
        "--split",      split,
        "--model_name", QA_MODEL,
        "--port",       VLLM_URL,
    ], cwd=REPO_DIR)

Выходные данные были обрезаны до нескольких последних строк (5000).
using vllm:  20%|██        | 2/10 [00:01<00:03,  2.18it/s]

using vllm: 100%|██████████| 10/10 [00:01<00:00,  7.56it/s]


using vllm:   0%|          | 0/4 [00:00<?, ?it/s]INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


using vllm:  25%|██▌       | 1/4 [00:01<00:03,  1.04s/it]

using vllm: 100%|██████████| 4/4 [00:01<00:00,  3.48it/s]


using vllm:   0%|          | 0/2 [00:00<?, ?it/s]INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


using vllm:  50%|█████     | 1/2 [00:00<00:00,  1.83it/s]INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/cha

In [87]:
# Скачивание нужных папок (только если они существуют)
import os
import shutil
from google.colab import files

REPO_DIR = "/content/verbal_uncertainty_feature_calibration"

folders_to_download = [
    "sem_uncertainty/outputs",
    "verbal_uncertainty/outputs",
    "datasets/nq_open",
    "calibration/outputs",
    "detection/LR_outputs",
    "outputs",
]

created_archives = []

for rel_path in folders_to_download:
    abs_path = os.path.join(REPO_DIR, rel_path)
    if os.path.isdir(abs_path):
        archive_base = f"/content/{rel_path.replace('/', '__')}"
        archive_path = shutil.make_archive(archive_base, "zip", abs_path)
        created_archives.append(archive_path)
        print(f"[OK] Архив создан: {archive_path}")
    else:
        print(f"[SKIP] Папка не найдена: {abs_path}")

print("\nНачинаю скачивание архивов...")
for archive_path in created_archives:
    print(f"Downloading: {archive_path}")
    files.download(archive_path)

print("\nГотово.")

[OK] Архив создан: /content/sem_uncertainty__outputs.zip
[OK] Архив создан: /content/verbal_uncertainty__outputs.zip
[OK] Архив создан: /content/datasets__nq_open.zip
[OK] Архив создан: /content/calibration__outputs.zip
[OK] Архив создан: /content/detection__LR_outputs.zip
[SKIP] Папка не найдена: /content/verbal_uncertainty_feature_calibration/outputs

Начинаю скачивание архивов...
Downloading: /content/sem_uncertainty__outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: /content/verbal_uncertainty__outputs.zip


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Downloading: /content/detection__LR_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Готово.


### 6.2 LLM-as-a-Judge for verbal uncertainty

**Details:** the judge model examines each of 10 answers (with uncertainty prompt) and assigns a "decisiveness score" from 0 (full hedging) to 1 (full confidence). VU for a question = mean over 10 answers.  
Output → `{model}_{dataset}_{split}_uncertainty_1.0_vu-llm-judge.json`.

**Paper §2.2, Appendix A.2** — Verbal Uncertainty Judge Prompt

In [63]:
path = "/content/verbal_uncertainty_feature_calibration/verbal_uncertainty/vu_llm_judge.py"
text = open(path).read()
old = """    print('len(all_message)', len(all_message))
    print('history_i', history_i)
    print('N', N)
    if len(all_message) == 0:
        return
"""
new = """    print('len(all_message)', len(all_message))
    print('history_i', history_i)
    if len(all_message) == 0:
        print(f"Nothing to do: checkpoint has {history_i} questions, dataset has {len(qa_ds)} — split already complete.")
        return
    print('N', N)
"""
if old not in text:
    raise RuntimeError("Pattern not found — file already patched or different version.")
open(path, "w").write(text.replace(old, new))
print("patched vu_llm_judge.py")

RuntimeError: Pattern not found — file already patched or different version.

In [58]:
for split in SPLITS:
    print(f"\n=== vu_llm_judge | split={split} ===")
    run([
        sys.executable, "verbal_uncertainty/vu_llm_judge.py",
        "--results_dir",   "outputs",
        "--dataset",       DATASET,
        "--split",         split,
        "--model_name",    QA_MODEL,
        "--prompt_method", "uncertainty",
        "--temperature",   "1.0",
        "--port",          VLLM_URL,
        "--batch_size",    "20",
    ], cwd=REPO_DIR)


=== vu_llm_judge | split=train ===
2026-03-23 21:10:12.012904: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774300212.035290  171708 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774300212.042632  171708 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774300212.061783  171708 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774300212.061805  171708 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774300212.061807  171708 computation_p

### 6.3 Label answer accuracy

**Details:** the judge model compares the greedy answer (T=0.1) against gold answers and returns 0 or 1.  
Output → `test_most_likely_acc.json`. Checkpoint every 400 records.

**Paper §2.3, Appendix A.3** — Accuracy Judge Prompt: "Does the proposed answer mean the same as any of the expected answers?"

In [61]:
for split in SPLITS:
    print(f"\n=== eval_acc | split={split} ===")
    run([
        sys.executable, "hallu_labeling/eval_acc.py",
        "--dataset",    DATASET,
        "--split",      split,
        "--model_name", QA_MODEL,
        "--port",       VLLM_URL,
    ], cwd=REPO_DIR)


=== eval_acc | split=train ===
2026-03-23 21:14:44.618154: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774300484.641147  174516 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774300484.648639  174516 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774300484.667584  174516 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774300484.667607  174516 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774300484.667609  174516 computation_place

### 6.4 Label refusals (refusal rate)

**Details:** the judge model checks whether the QA model refused to answer.  
Final label: `hallucinated` if acc=0 AND refusal=False; `ok` otherwise.  
Output → `test_refusal_rate.json`. Checkpoint every 100 records (resumes on restart).

**Paper §2.3** — "samples where the model does not refuse and the answer is not entailed by the golden answer are labeled as hallucinations"

In [66]:
for split in SPLITS:
    print(f"\n=== get_refusal_rate | split={split} ===")
    run([
        sys.executable, "hallu_labeling/get_refusal_rate.py",
        "--dataset",    DATASET,
        "--split",      split,
        "--model_name", QA_MODEL,
        "--port",       VLLM_URL,
    ], cwd=REPO_DIR)


=== get_refusal_rate | split=train ===
2026-03-23 21:17:44.658311: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774300664.681673  175874 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774300664.689731  175874 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774300664.709195  175874 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774300664.709228  175874 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774300664.709231  175874 computati

### 6.5 Stop vLLM

In [68]:
if vllm_proc is not None:
    vllm_proc.terminate()
    vllm_proc.wait()
    gc.collect()
    torch.cuda.empty_cache()
    print(f"vLLM остановлен. GPU: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB свободно")
else:
    print("vLLM не запускался (USE_OPENROUTER=True).")

vLLM не запускался (USE_OPENROUTER=True).


---
# Phase 3 — Data Merge

### 7. Merge

**Details:** merges three sources into a single CSV:
- `test_0.1.jsonl` — greedy model answers
- `test_most_likely_acc.json` — accuracy labels (0/1)
- `test_refusal_rate.json` — refusal labels (True/False)
- `vu-llm-judge.json` — verbal uncertainty scores
- `test_semantic_entropy.pkl` — semantic entropy

Final label: `ok` (acc=1 OR refusal=True) / `hallucinated` (acc=0 AND refusal=False).  
Output → `datasets/nq_open/Mistral-7B-Instruct-v0.3_sentence/test.csv`.

**Paper §2.3, Figure 2** — miscalibration between high SU and low VU

In [69]:
# Диагностика длин перед merge — проверяем все сплиты
import json, jsonlines, pandas as pd

sampled_dir = f"{REPO_DIR}/datasets/{DATASET}/sampled"
base = f"{REPO_DIR}/sem_uncertainty/outputs/{DATASET}/sentence/{QA_MODEL}"

for split in SPLITS:
    csv_path = f"{sampled_dir}/{split}.csv"
    n_csv = len(pd.read_csv(csv_path)) if os.path.exists(csv_path) else "?"
    print(f"\n── {split} (CSV: {n_csv} строк) ──────────────────────")
    for fname in [f"{split}_0.1.jsonl", f"{split}_refusal_rate.json", f"{split}_most_likely_acc.json"]:
        path = f"{base}/{fname}"
        if not os.path.exists(path):
            print(f"  MISSING : {fname}")
            continue
        if fname.endswith(".jsonl"):
            with jsonlines.open(path) as f:
                n = sum(1 for _ in f)
        elif "refusal" in fname:
            n = len(json.load(open(path)).get("refusal", []))
        else:
            n = len(json.load(open(path)))
        status = "✓" if n == n_csv else f"⚠ MISMATCH (ожидалось {n_csv})"
        print(f"  {fname}: {n} записей  {status}")


── train (CSV: 500 строк) ──────────────────────
  train_0.1.jsonl: 500 записей  ✓
  train_refusal_rate.json: 500 записей  ✓
  train_most_likely_acc.json: 500 записей  ✓

── test (CSV: 500 строк) ──────────────────────
  test_0.1.jsonl: 500 записей  ✓
  test_refusal_rate.json: 500 записей  ✓
  test_most_likely_acc.json: 500 записей  ✓


In [70]:
# Если refusal_rate.json < ожидаемого — дополнить False для каждого сплита
# (NQ-Open: Mistral редко отказывается от ответа)
import pandas as pd, json, numpy as np

sampled_dir = f"{REPO_DIR}/datasets/{DATASET}/sampled"
base = f"{REPO_DIR}/sem_uncertainty/outputs/{DATASET}/sentence/{QA_MODEL}"

for split in SPLITS:
    refusal_path = f"{base}/{split}_refusal_rate.json"
    csv_path = f"{sampled_dir}/{split}.csv"
    if not os.path.exists(refusal_path):
        print(f"[{split}] refusal_rate.json не найден — пропускаем")
        continue
    n_expected = len(pd.read_csv(csv_path))
    data = json.load(open(refusal_path))
    n = len(data["refusal"])
    if n < n_expected:
        data["refusal"] = data["refusal"] + [False] * (n_expected - n)
        data["refusal_rate"] = float(np.mean(data["refusal"]))
        json.dump(data, open(refusal_path, "w"), indent=4)
        print(f"[{split}] refusal дополнен: {n} → {n_expected}")
    else:
        print(f"[{split}] refusal OK: {n} записей")

[train] refusal OK: 500 записей
[test] refusal OK: 500 записей


In [72]:
# Обрезка артефактов до первых N примеров (индексы 0..N-1). Перезаписывает файлы — при необходимости сделай копию.
import json
import pickle
from pathlib import Path
import pandas as pd
import jsonlines

N = 500
REPO_DIR = "/content/verbal_uncertainty_feature_calibration"
DATASET = "nq_open"
QA_MODEL = "Mistral-7B-Instruct-v0.3"
SPLITS = ["train", "test"]

root = Path(REPO_DIR)
sem_base = root / "sem_uncertainty" / "outputs" / DATASET / "sentence" / QA_MODEL
vu_out = root / "verbal_uncertainty" / "outputs"
sampled_dir = root / "datasets" / DATASET / "sampled"
ds_model = root / "datasets" / DATASET / QA_MODEL
ds_sentence = root / "datasets" / DATASET / f"{QA_MODEL}_sentence"

def trunc_jsonl(p, n):
    if not p.is_file():
        return
    rows = []
    with jsonlines.open(p) as r:
        for i, o in enumerate(r):
            if i >= n:
                break
            rows.append(o)
    with jsonlines.open(p, "w") as w:
        for o in rows:
            w.write(o)
    print("ok jsonl", p)

def trunc_vu_judge(p, n):
    if not p.is_file():
        return
    d = json.load(open(p))
    if isinstance(d, list) and len(d) > n:
        json.dump(d[:n], open(p, "w"))
    print("ok judge", p)

def trunc_refusal(p, n):
    if not p.is_file():
        return
    d = json.load(open(p))
    r = d.get("refusal", [])
    if len(r) > n:
        d["refusal"] = r[:n]
        d["refusal_rate"] = float(sum(d["refusal"]) / len(d["refusal"])) if d["refusal"] else 0.0
        json.dump(d, open(p, "w"), indent=4)
    print("ok refusal", p)

def trunc_acc(p, ids_keep):
    if not p.is_file() or not ids_keep:
        return
    acc = json.load(open(p))
    keep = set(ids_keep)
    acc = {k: v for k, v in acc.items() if str(k) in keep}
    json.dump(acc, open(p, "w"))
    print("ok acc", p)

def trunc_pkl(p, n):
    if not p.is_file():
        return
    o = pickle.load(open(p, "rb"))
    e = o["uncertainty_measures"]["cluster_assignment_entropy"]
    s = o["semantic_ids"]
    if len(e) > n:
        o["uncertainty_measures"]["cluster_assignment_entropy"] = e[:n]
        o["semantic_ids"] = s[:n]
        pickle.dump(o, open(p, "wb"))
    print("ok pkl", p)

def trunc_csv(p, n):
    if not p.is_file():
        return
    df = pd.read_csv(p)
    if len(df) > n:
        df.head(n).to_csv(p, index=False)
    print("ok csv", p)

for split in SPLITS:
    print(f"\n--- {split} ---")
    sp = sampled_dir / f"{split}.csv"
    ids = []
    if sp.is_file():
        df0 = pd.read_csv(sp)
        ids = df0["id"].astype(str).tolist()[:N]
        trunc_csv(sp, N)

    trunc_jsonl(sem_base / f"{split}_1.0.jsonl", N)
    trunc_jsonl(sem_base / f"{split}_0.1.jsonl", N)
    trunc_pkl(sem_base / f"{split}_semantic_entropy.pkl", N)
    trunc_refusal(sem_base / f"{split}_refusal_rate.json", N)
    trunc_acc(sem_base / f"{split}_most_likely_acc.json", ids)

    trunc_vu_judge(vu_out / f"{QA_MODEL}_{DATASET}_{split}_uncertainty_1.0_vu-llm-judge.json", N)
    trunc_jsonl(vu_out / f"{QA_MODEL}_{DATASET}_{split}_uncertainty_1.0.jsonl", N)

    trunc_csv(ds_model / f"{split}.csv", N)
    trunc_csv(ds_sentence / f"{split}.csv", N)

print("\nГотово. Запусти merge заново.")


--- train ---
ok csv /content/verbal_uncertainty_feature_calibration/datasets/nq_open/sampled/train.csv
ok jsonl /content/verbal_uncertainty_feature_calibration/sem_uncertainty/outputs/nq_open/sentence/Mistral-7B-Instruct-v0.3/train_1.0.jsonl
ok jsonl /content/verbal_uncertainty_feature_calibration/sem_uncertainty/outputs/nq_open/sentence/Mistral-7B-Instruct-v0.3/train_0.1.jsonl
ok pkl /content/verbal_uncertainty_feature_calibration/sem_uncertainty/outputs/nq_open/sentence/Mistral-7B-Instruct-v0.3/train_semantic_entropy.pkl
ok refusal /content/verbal_uncertainty_feature_calibration/sem_uncertainty/outputs/nq_open/sentence/Mistral-7B-Instruct-v0.3/train_refusal_rate.json
ok acc /content/verbal_uncertainty_feature_calibration/sem_uncertainty/outputs/nq_open/sentence/Mistral-7B-Instruct-v0.3/train_most_likely_acc.json
ok judge /content/verbal_uncertainty_feature_calibration/verbal_uncertainty/outputs/Mistral-7B-Instruct-v0.3_nq_open_train_uncertainty_1.0_vu-llm-judge.json
ok jsonl /conte

In [74]:
import os
import re

REPO_DIR = "/content/verbal_uncertainty_feature_calibration"

# ── 1. src/eval_utils.py ─────────────────────────────────────────────────────
with open(f"{REPO_DIR}/src/eval_utils.py", "w") as f:
    f.write('''
import os, sys, openai
from tqdm.contrib.concurrent import thread_map

current_dir = os.path.dirname(os.path.abspath(__file__))
root_path = os.path.dirname(current_dir)
sys.path.append(f"{root_path}/sem_uncertainty/")
from semantic_entropy.huggingface_models import HuggingfaceModel


class VLLM:
    def __init__(self, name, max_new_tokens):
        self.port = name
        self.max_tokens = max_new_tokens
        self.openrouter_model_id = os.environ.get(
            "OPENROUTER_MODEL_ID",
            "meta-llama/llama-3.1-70b-instruct",
        )
        self.vllm_model_id = "meta-llama/Llama-3.1-70B-Instruct"

    def predict(self, prompt, temperature, output_hidden_states=False):
        is_openrouter = ("openrouter.ai" in self.port) or ("openrouter" in self.port)
        api_key = os.environ.get("OPENROUTER_API_KEY") if is_openrouter else "NOT A REAL KEY"
        model_id = self.openrouter_model_id if is_openrouter else self.vllm_model_id
        client = openai.OpenAI(base_url=self.port, api_key=api_key)
        resp = client.chat.completions.create(
            model=model_id,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=self.max_tokens,
            temperature=temperature,
        )
        return resp.choices[0].message.content, "", ""

    def batch_predict(self, batch_prompts, temperature, output_hidden_states=False):
        return thread_map(
            lambda p: self.predict(p, temperature=temperature),
            batch_prompts, max_workers=20, desc="vllm",
        )


def get_available_servers():
    vllm_url = os.environ.get("VLLM_URL", "http://localhost:8000/v1")
    return {
        "meta-llama/Llama-3.1-70B-Instruct": {
            "server_urls": [vllm_url],
            "job_ids": ["0"],
        }
    }
''')
print("[1/9] eval_utils.py OK")

# ── 2. generate_answers.py ────────────────────────────────────────────────────
path = f"{REPO_DIR}/sem_uncertainty/generate_answers.py"
code = open(path).read()
old = "        inputs = prepare_inputs(tokenizer, batch_local_prompt)\n\n        with torch.no_grad():"
new = "        inputs = prepare_inputs(tokenizer, batch_local_prompt)\n        inputs = inputs.to(device)\n\n        with torch.no_grad():"
if old in code:
    open(path, "w").write(code.replace(old, new))
    print("[2/9] generate_answers.py OK")
else:
    print("[2/9] generate_answers.py — уже пропатчен")

# ── 3–4. merge.py / merge_vuf.py — default train+test, без val ───────────────
def _patch_default_splits_py(path, tag):
    code = open(path).read()
    orig = code
    pairs = [
        ("splits = args.splits if args.splits else ['test', 'val', 'train']",
         "splits = args.splits if args.splits else ['train', 'test']"),
        ('splits = args.splits if args.splits else ["test", "val", "train"]',
         "splits = args.splits if args.splits else ['train', 'test']"),
        ("splits = args.splits if args.splits else ['val', 'test', 'train']",
         "splits = args.splits if args.splits else ['train', 'test']"),
        ('splits = args.splits if args.splits else ["val", "test", "train"]',
         "splits = args.splits if args.splits else ['train', 'test']"),
        ("for split in ['test', 'val', 'train']:", "for split in ['train', 'test']:"),
        ('for split in ["test", "val", "train"]:', "for split in ['train', 'test']:"),
        ("for split in ['val', 'test', 'train']:", "for split in ['train', 'test']:"),
        ('for split in ["val", "test", "train"]:', "for split in ['train', 'test']:"),
    ]
    for a, b in pairs:
        code = code.replace(a, b)
    code, _ = re.subn(
        r"splits = args\.splits if args\.splits else \[(?:'|\")(?:test|val|train)(?:'|\"),\s*(?:'|\")(?:test|val|train)(?:'|\"),\s*(?:'|\")(?:test|val|train)(?:'|\")\]",
        "splits = args.splits if args.splits else ['train', 'test']",
        code,
        count=1,
    )
    if code != orig:
        open(path, "w").write(code)
        print(f"{tag} — обновлён → default ['train','test']")
    else:
        print(f"{tag} — без изменений")

_patch_default_splits_py(f"{REPO_DIR}/datasets/merge.py", "[3/9] datasets/merge.py")
_patch_default_splits_py(f"{REPO_DIR}/calibration/merge_vuf.py", "[4/9] merge_vuf.py")

# ── 5. LogisticRegression.py ─────────────────────────────────────────────────
path = f"{REPO_DIR}/detection/LogisticRegression.py"
code = open(path).read()
if "train_split = 'test'" in code:
    open(path, "w").write(code.replace("train_split = 'test'", "train_split = 'train'"))
    print("[5/9] LogisticRegression — train_split='train'")
else:
    print("[5/9] LogisticRegression — OK")

# ── 6. causal.py ────────────────────────────────────────────────────────────
path = f"{REPO_DIR}/calibration/causal.py"
code = open(path).read()
old_a = '        device_idx_l = model.hf_device_map[f"model.layers.{l}"]'
new_a = '        device_idx_l = model.hf_device_map.get(f"model.layers.{l}", model.hf_device_map.get("", 0))'
code = code.replace(old_a, new_a)
old_b = """                else:
                    assert False
                    if outputs.shape[1] > 1:
                        outputs += h_feature_l * alpha

                    return outputs"""
new_b = """                else:
                    if outputs.shape[1] > 1:
                        outputs = outputs + h_feature_l * alpha
                    return outputs"""
code = code.replace(old_b, new_b)
old_c = '    else:\n        output_dir = f"outputs/{dataset}/{model_name}/{prompt_type}/{split}"\n    if args.run_certain:'
new_c = '    else:\n        output_dir = f"outputs/{dataset}/{model_name}/{prompt_type}/{split}"\n    os.makedirs(output_dir, exist_ok=True)\n    if args.run_certain:'
code = code.replace(old_c, new_c)
open(path, "w").write(code)
print("[6/9] causal.py OK")

# ── 7. vu_llm_judge.py — OpenRouter ───────────────────────────────────────────
path = f"{REPO_DIR}/verbal_uncertainty/vu_llm_judge.py"
code = open(path).read()
if 'api_key="NOT A REAL KEY"' in code:
    code = code.replace(
        'api_key="NOT A REAL KEY"',
        'api_key=os.environ.get("OPENROUTER_API_KEY") if "openrouter.ai" in judge_model else "NOT A REAL KEY"',
    )
    code = code.replace(
        "model='meta-llama/Llama-3.1-70B-Instruct'",
        'model=os.environ.get("OPENROUTER_MODEL_ID","meta-llama/llama-3.1-70b-instruct") if "openrouter.ai" in judge_model else \'meta-llama/Llama-3.1-70B-Instruct\'',
    )
    open(path, "w").write(code)
    print("[7/9] vu_llm_judge.py OK")
else:
    print("[7/9] vu_llm_judge.py — уже пропатчен")

# ── 8. refusal.py ─────────────────────────────────────────────────────────────
path = f"{REPO_DIR}/src/refusal.py"
code = open(path).read()
if 'api_key="NOT A REAL KEY"' in code:
    code = code.replace(
        'api_key="NOT A REAL KEY"',
        'api_key=os.environ.get("OPENROUTER_API_KEY") if "openrouter.ai" in str(port) else "NOT A REAL KEY"',
    )
    code = code.replace(
        "model='meta-llama/Llama-3.1-70B-Instruct'",
        'model=os.environ.get("OPENROUTER_MODEL_ID","meta-llama/llama-3.1-70b-instruct") if "openrouter.ai" in str(port) else \'meta-llama/Llama-3.1-70B-Instruct\'',
    )
    open(path, "w").write(code)
    print("[8/9] refusal.py OK")
else:
    print("[8/9] refusal.py — уже пропатчен")

# ── 9. semantic_entropy.py — OpenRouter + таймауты/workers ────────────────────
path = f"{REPO_DIR}/sem_uncertainty/semantic_entropy/semantic_entropy.py"
code = open(path).read()
if 'api_key="NOT A REAL KEY"' in code:
    code = code.replace(
        'api_key="NOT A REAL KEY"',
        'api_key=os.environ.get("OPENROUTER_API_KEY") if "openrouter.ai" in self.port else "NOT A REAL KEY"',
    )
    code = code.replace(
        "model=self.name",
        'model=os.environ.get("OPENROUTER_MODEL_ID","meta-llama/llama-3.1-70b-instruct") if "openrouter.ai" in self.port else self.name',
    )
    code = code.replace(
        "max_tokens=30,\n            temperature=temperature,",
        "max_tokens=int(os.environ.get('OPENROUTER_ENTAILMENT_MAX_TOKENS', '8')),\n            temperature=temperature,\n            request_timeout=int(os.environ.get('OPENROUTER_REQUEST_TIMEOUT', '30')),",
    )
    code = code.replace(
        "max_workers=20,",
        "max_workers=int(os.environ.get('SEM_ENTROPY_MAX_WORKERS', '40')),",
    )
    open(path, "w").write(code)
    print("[9/9] semantic_entropy.py OK")
else:
    print("[9/9] semantic_entropy.py — уже пропатчен")

print("\nГотово: все патчи применены.")

[1/9] eval_utils.py OK
[2/9] generate_answers.py — уже пропатчен
[3/9] datasets/merge.py — обновлён → default ['train','test']
[4/9] merge_vuf.py — обновлён → default ['train','test']
[5/9] LogisticRegression — OK
[6/9] causal.py OK
[7/9] vu_llm_judge.py — уже пропатчен
[8/9] refusal.py — уже пропатчен
[9/9] semantic_entropy.py — уже пропатчен

Готово: все патчи применены.


In [79]:
!{sys.executable} datasets/merge.py --dataset {DATASET} --model {QA_MODEL}

refusal rate 0.026
100% 500/500 [00:00<00:00, 10220.69it/s]
100% 500/500 [00:00<00:00, 13403.59it/s]
100% 500/500 [00:00<00:00, 18312.86it/s]
refusal rate 0.05
100% 500/500 [00:00<00:00, 10665.80it/s]
100% 500/500 [00:00<00:00, 13709.21it/s]
100% 500/500 [00:00<00:00, 18329.34it/s]


In [81]:
import pandas as pd
base = f"{REPO_DIR}/datasets/{DATASET}/{QA_MODEL}"
sent = f"{REPO_DIR}/datasets/{DATASET}/{QA_MODEL}_sentence"
for sp in ["train", "test"]:
    m = pd.read_csv(f"{base}/{sp}.csv")
    s = pd.read_csv(f"{sent}/{sp}.csv")[["id", "model_generated"]]
    m = m.drop(columns=["model_generated"], errors="ignore").merge(s, on="id", how="left")
    m.to_csv(f"{base}/{sp}.csv", index=False)
    print(sp, "model_generated ok:", m["model_generated"].notna().mean())

train model_generated ok: 1.0
test model_generated ok: 1.0


In [82]:
# ---- Save hidden states (activations) for train/val/test ----
import shutil

# Some scripts expect datasets without the `_sentence` suffix.
src_dir = f"{REPO_DIR}/datasets/{DATASET}/{QA_MODEL}_sentence"
dst_dir = f"{REPO_DIR}/datasets/{DATASET}/{QA_MODEL}"
os.makedirs(dst_dir, exist_ok=True)

for split in SPLITS:
    src_csv = f"{src_dir}/{split}.csv"
    dst_csv = f"{dst_dir}/{split}.csv"
    if os.path.exists(src_csv) and not os.path.exists(dst_csv):
        shutil.copyfile(src_csv, dst_csv)
        print(f"Copied: {src_csv} -> {dst_csv}")
    else:
        print(f"Skip copy {split}: src_exists={os.path.exists(src_csv)} dst_exists={os.path.exists(dst_csv)}")

# NOTE: get_hidden_state.py извлекает hidden states для `model_generated` (по 1 ответу на вопрос).
# Если нужно сохранять hidden states для всех 10 сэмплов ответа, это требует отдельного изменения merge/probe.
SAVE_HIDDEN_STATES = True
LAYERS_TO_PROCESS_EXPR = "range(15,32)"
INFO_TYPE = "last"
SAVE_CACHE = "pipeline_used_layers_last"

if SAVE_HIDDEN_STATES:
    print(f"\nExtracting activations: layers={LAYERS_TO_PROCESS_EXPR} info_type={INFO_TYPE} splits={SPLITS}")
    run([
        sys.executable, "probe/get_hidden_state.py",
        "--source_dirs",        dst_dir,
        "--layers_to_process",  LAYERS_TO_PROCESS_EXPR,
        "--info_type",          INFO_TYPE,
        "--save_hidden_state",
        "--internal_model_name", QA_MODEL,
        "--splits",             *SPLITS,
        "--save_cache",        SAVE_CACHE,
        "--save_dir_root",     "tmp",
    ], cwd=REPO_DIR)


Skip copy train: src_exists=True dst_exists=True
Skip copy test: src_exists=True dst_exists=True

Extracting activations: layers=range(15,32) info_type=last splits=['train', 'test']
/content/verbal_uncertainty_feature_calibration/probe/get_hidden_state.py:530: SyntaxWarning: invalid escape sequence '\/'
  args.save_dir_root = re.sub("\/+", "/", args.save_dir_root)
2026-03-23 21:48:26.111325: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774302506.134182  184118 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774302506.141843  184118 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774302506.160675  184118 computation_placer.cc:177] compu

---
# Phase 4 — Extract VUF (Verbal Uncertainty Feature)

### 8.1 Compute VUF

**Details:** runs questions through Mistral and saves last-token hidden states on layers 15–32.  
VUF = normalized difference-in-means between `D_uncertain` (VU ≥ 0.9) and `D_certain` (VU ≤ 0.05).  
Output → `calibration/outputs/nq_open/{model}/{prompt_type}/test/Hs_hedge_universal.pt`.

**Paper §3.1, equations (2)–(3)** — difference-in-means, L2-normalization

In [84]:
for split in SPLITS:
    for prompt_type in ["uncertainty", "sentence"]:
        print(f"\n=== universal_vuf | split={split} | prompt={prompt_type} ===")
        run([
            sys.executable, "calibration/universal_vuf.py",
            "--dataset",     DATASET,
            "--split",       split,
            "--model_name",  QA_MODEL,
            "--prompt_type", prompt_type,
        ], cwd=REPO_DIR)


=== universal_vuf | split=train | prompt=uncertainty ===
2026-03-23 21:52:46.286155: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774302766.308702  185309 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774302766.316225  185309 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774302766.335178  185309 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774302766.335201  185309 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774302766.335203

### 8.2 Merge VUF features

**Details:** merges `.pt` files into a single tensor.  
Output → `calibration/outputs/merged/{model}/{prompt_type}/Hs_hedge_universal.pt`.

**Paper §3.2** — "VUFs are consistent across different datasets"

In [85]:
for prompt_type in ["uncertainty", "sentence"]:
    print(f"\n=== merge_vuf | prompt={prompt_type} ===")
    run([
        sys.executable, "calibration/merge_vuf.py",
        "--model_name",  QA_MODEL,
        "--prompt_type", prompt_type,
        "--datasets",    DATASET,
    ], cwd=REPO_DIR)


=== merge_vuf | prompt=uncertainty ===
dataset: nq_open
13 torch.Size([13, 32, 4096])
329 torch.Size([329, 32, 4096])
22 torch.Size([22, 32, 4096])
286 torch.Size([286, 32, 4096])
all_Hs_questions_uncertain_verbal torch.Size([35, 32, 4096])
all_Hs_questions_certain_verbal torch.Size([615, 32, 4096])

=== merge_vuf | prompt=sentence ===
dataset: nq_open
13 torch.Size([13, 32, 4096])
329 torch.Size([329, 32, 4096])
22 torch.Size([22, 32, 4096])
286 torch.Size([286, 32, 4096])
all_Hs_questions_uncertain_verbal torch.Size([35, 32, 4096])
all_Hs_questions_certain_verbal torch.Size([615, 32, 4096])


In [88]:
# LogisticRegression обучается на train-сплите, предсказывает на predict_split.
# Для разработки используй predict_split="val", для финального отчёта — "test".
for predict_split in ["test"]:
    print(f"\n{'='*50}")
    print(f"  predict_split = {predict_split}")
    print('='*50)
    run([
        sys.executable, "detection/LogisticRegression.py",
        "--model_name",    QA_MODEL,
        "--dataset",       DATASET,
        "--predict_split", predict_split,
    ], cwd=REPO_DIR)

print("\nФормат: строка 1 = только SU | строка 2 = только VU | строка 3 = VU+SU")
print("Колонки: AUROC  Accuracy")


  predict_split = test
Training on nq_open
72.07	65.8
65.38	60.4
72.35	65.4

Формат: строка 1 = только SU | строка 2 = только VU | строка 3 = VU+SU
Колонки: AUROC  Accuracy


In [89]:
!{sys.executable} detection/LogisticRegression.py \
    --model_name    {QA_MODEL} \
    --dataset       {DATASET} \
    --predict_split test

print("\nФормат: строка 1 = только SU | строка 2 = только VU | строка 3 = VU+SU")
print("Колонки: AUROC  Accuracy")

Training on nq_open
72.07	65.8
65.38	60.4
72.35	65.4

Формат: строка 1 = только SU | строка 2 = только VU | строка 3 = VU+SU
Колонки: AUROC  Accuracy


---
# Phase 6 — Causal Validation of VUF

**Details:** runs questions through Mistral with modified hidden states:  
`h^(l)(x) ← h^(l)(x) + α * r_VUF^(l)` for layers 15–32.

- **run_certain** (α > 0): confident questions (VU ≤ 0.05) → add VUF → model hedges  
- **run_uncertain** (α < 0): uncertain questions (VU ≥ 0.9) → subtract VUF → model becomes more confident

**Paper §3.2, Figure 5, equation (4)** — causal validation of VUF

In [ ]:
import numpy as np

# run_certain: α = 0.25 .. 2.0
for alpha in np.arange(0.25, 2.25, 0.25):
    alpha_r = round(float(alpha), 2)
    print(f"\n=== causal run_certain | α={alpha_r} ===")
    run([
        sys.executable, "calibration/causal.py",
        "--dataset",            DATASET,
        "--split",              "test",
        "--model_name",         QA_MODEL,
        "--prompt_type",        "sentence",
        "--str_process_layers", "range(15,32)",
        "--run_certain",        "1",
        "--iti_method",         "2",
        "--alpha",              str(alpha_r),
    ], cwd=REPO_DIR)

In [ ]:
# run_uncertain: α = -0.25 .. -2.0
for alpha in np.arange(0.25, 2.25, 0.25):
    alpha_r = -round(float(alpha), 2)
    print(f"\n=== causal run_uncertain | α={alpha_r} ===")
    run([
        sys.executable, "calibration/causal.py",
        "--dataset",            DATASET,
        "--split",              "test",
        "--model_name",         QA_MODEL,
        "--prompt_type",        "sentence",
        "--str_process_layers", "range(15,32)",
        "--run_uncertain",      "1",
        "--iti_method",         "2",
        "--alpha",              str(alpha_r),
    ], cwd=REPO_DIR)

---
# Phase 7 — Mechanistic Uncertainty Calibration (MUC)

### 10.1 Semantic Control

**Details:** for each question flagged by the detector as a potential hallucination, the intervention strength is computed as:  
`α_su(x) = clip(SU_norm(x) − VU(x), 0, max_α)`  
Applies VUF intervention only where SU > VU (the model is semantically uncertain but verbally confident).  

**Paper §4.2, equations (5)–(6)** — Mechanistic Uncertainty Calibration (MUC)

In [ ]:
print("\n=== semantic_control | split=test ===")
run([
    sys.executable, "calibration/semantic_control.py",
    "--dataset",            DATASET,
    "--split",              "test",
    "--model_name",         QA_MODEL,
    "--prompt_type",        "uncertainty",
    "--str_process_layers", "range(15,32)",
    "--iti_method",         "2",
    "--max_alpha",          "1.0",
    "--use_predicted",      "0",
], cwd=REPO_DIR)

### 10.2 Start vLLM for MUC evaluation

In [ ]:
vllm_proc = None

if not USE_OPENROUTER:
    gc.collect()
    torch.cuda.empty_cache()

    vllm_proc = subprocess.Popen(
        [
            sys.executable, "-m", "vllm.entrypoints.openai.api_server",
            "--model",                  JUDGE_HF_ID,
            "--served-model-name",      JUDGE_ALIAS,
            "--port",                   str(VLLM_PORT),
            "--gpu-memory-utilization", "0.85",
            "--max-model-len",          "4096",
            "--dtype",                  "float16",
        ],
        stdout=open(VLLM_LOG, "w"),
        stderr=subprocess.STDOUT,
        cwd=REPO_DIR,
    )
    print(f"vLLM PID={vllm_proc.pid}...")
    for i in range(120):
        try:
            if requests.get(f"http://localhost:{VLLM_PORT}/health", timeout=3).status_code == 200:
                print(f"vLLM готов ({i*5}s)")
                break
        except Exception:
            pass
        time.sleep(5)
else:
    print("USE_OPENROUTER=True — пропускаем запуск vLLM сервера (используем OpenRouter).")

### 10.3 Evaluate MUC results

**Details:** three eval scripts run the judge on `semantic_control.py` outputs and compute:  
- `eval_acc.py` — whether answer accuracy changed after intervention  
- `eval_vu.py` — whether verbal uncertainty changed (did the model start hedging?)  
- `compute_semantic_entropy.py` — semantic entropy of the new answers

**Paper §4.2, Table 3** — Mitigation Results: Hallucination Rate ↓, VU/SU Disagreement Rate ↓, Correlation ↑

In [ ]:
COMMON = [
    "--dataset",            DATASET,
    "--split",              "test",
    "--model_name",         QA_MODEL,
    "--prompt_type",        "uncertainty",
    "--str_process_layers", "range(15,32)",
    "--iti_method",         "2",
    "--max_alpha",          "1.0",
]

for script, extra in [
    ("calibration/eval/eval_acc.py",               ["--entailment_model", VLLM_URL]),
    ("calibration/eval/eval_vu.py",                ["--entailment_model", VLLM_URL]),
    ("calibration/eval/compute_semantic_entropy.py", ["--entailment_model", VLLM_URL]),
]:
    if os.path.exists(f"{REPO_DIR}/{script}"):
        print(f"\n=== {script} ===")
        run(
            [sys.executable, script] + COMMON + extra,
            cwd=REPO_DIR,
        )
    else:
        print(f"Пропущен: {script}")

In [ ]:
if vllm_proc is not None:
    vllm_proc.terminate()
    vllm_proc.wait()
    gc.collect()
    torch.cuda.empty_cache()
    print("vLLM остановлен. Пайплайн завершён.")
else:
    print("vLLM не запускался (USE_OPENROUTER=True).")

---
## Troubleshooting

| Problem | Solution |
|---|---|
| `CUDA out of memory` | Restart runtime or reduce `batch_size` |
| vLLM won't start | `!tail -50 /tmp/vllm.log` |
| `KeyError: meta-llama/...` | vLLM is not running — re-run cell 6.0 |
| Length mismatch in merge | Run the refusal backfill cell (section 7) |
| `FileNotFoundError` in causal.py | Verify that merge (phase 3) and merge_vuf (phase 4) completed |
| HuggingFace 403 | Accept the license on HF Hub and re-check the token |

```python
# Check vLLM logs
!tail -50 /tmp/vllm.log
```